### Step 1A — Load LCMS and inspect the annual product structure

The USDA Forest Service Landscape Change Monitoring System (LCMS) is loaded from Google Earth Engine and restricted to the conterminous United States.

LCMS will be used primarily as a disturbance-quality-control source. For each focal fire, the project will examine a recent pre-fire disturbance-history window as well as the complete period from the focal fire through the fifth post-fire growing season.

For the Highline Fire proof of concept, a broad 10-year pre-fire audit window is retained for inspection:

- 2007–2016: pre-fire disturbance-history window
- 2017: focal wildfire year
- 2018–2022: post-fire recovery period through Year 5

The final production lookback length will not be fixed from Highline alone. Later population-level analysis will compare shorter 5-, 7-, and 10-year pre-fire disturbance windows.

In [1]:
import ee

ee.Initialize(project="ee-wildfire-recovery-vasant")

HIGHLINE_EVENT_ID = "ID4549011522720170729"

mtbs_boundaries = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

highline = ee.Feature(
    mtbs_boundaries
    .filter(ee.Filter.eq("Event_ID", HIGHLINE_EVENT_ID))
    .first()
)

highline_geom = highline.geometry()

print("Highline Event ID:", highline.get("Event_ID").getInfo())
print("Highline Fire name:", highline.get("Incid_Name").getInfo())

Highline Event ID: ID4549011522720170729
Highline Fire name: HIGHLINE


### Step 1A — Load LCMS and inspect the annual product structure

The current USDA Forest Service Landscape Change Monitoring System (LCMS) product is loaded from Google Earth Engine and restricted to the conterminous United States.

This first check confirms the annual time coverage, available bands, and basic metadata for the Highline study area before any disturbance classes are interpreted.

In [2]:
lcms = ee.ImageCollection(
    "projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11"
)

lcms_conus = lcms.filter(
    ee.Filter.eq("study_area", "CONUS")
)

lcms_highline = lcms_conus.filterBounds(highline_geom)

first_lcms = ee.Image(lcms_highline.sort("year").first())
last_lcms = ee.Image(lcms_highline.sort("year", False).first())

print("LCMS images intersecting Highline:",
      lcms_highline.size().getInfo())

print("First LCMS year:",
      first_lcms.get("year").getInfo())

print("Last LCMS year:",
      last_lcms.get("year").getInfo())

print("Bands:",
      first_lcms.bandNames().getInfo())

print("Study area:",
      first_lcms.get("study_area").getInfo())

print("LCMS version:",
      first_lcms.get("version").getInfo())

LCMS images intersecting Highline: 41
First LCMS year: 1985
Last LCMS year: 2025
Bands: ['Change', 'Change_Raw_Probability_Slow_Loss', 'Change_Raw_Probability_Fast_Loss', 'Change_Raw_Probability_Gain', 'Land_Cover', 'Land_Cover_Raw_Probability_Trees', 'Land_Cover_Raw_Probability_Tall-Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Trees-Mix', 'Land_Cover_Raw_Probability_Barren-and-Trees-Mix', 'Land_Cover_Raw_Probability_Tall-Shrubs', 'Land_Cover_Raw_Probability_Shrubs', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Barren-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb', 'Land_Cover_Raw_Probability_Barren-and-Grass-Forb-Herb-Mix', 'Land_Cover_Raw_Probability_Barren-or-Impervious', 'Land_Cover_Raw_Probability_Snow-or-Ice', 'Land_Cover_Raw_Probability_Water', 'Land_Use', 'Land_Use_Raw_Probability_Agriculture', 'Land_Use_Raw_Probability_Developed', 'Land_Use_Ra

#### Step 1A observations

The LCMS CONUS collection successfully covers the Highline Fire study area.

- 41 annual images are available from 1985 through 2025.
- The collection is the CONUS `v2025-11` release.
- The principal annual thematic products include `Change`, `Land_Cover`, and `Land_Use`, along with model-probability and QA bands.
- The available temporal record fully covers the Highline proof-of-concept audit window from 2007 through 2022.

LCMS will primarily be evaluated as a disturbance-quality-control source rather than as a large predictive feature family.

### Step 1B — Understand the LCMS change classes

The LCMS `Change` product provides an annual classification describing the likely cause of vegetation change at each 30 m pixel.

Before defining disturbance-screening rules, the change classes are grouped conceptually according to their potential role in the wildfire-recovery study.

The key distinction is between:

- the focal wildfire itself, which is expected and must not be treated as contamination,
- normal vegetation recovery or stability,
- major non-fire disturbances that could invalidate the interpretation of the recovery trajectory,
- and ambiguous natural disturbances that require careful treatment rather than automatic exclusion.

In [3]:
CHANGE_CLASSES = {
    1: "Wind",
    2: "Hurricane",
    3: "Snow or Ice Transition",
    4: "Desiccation",
    5: "Inundation",
    6: "Prescribed Fire",
    7: "Wildfire",
    8: "Mechanical Land Transformation",
    9: "Tree Removal",
    10: "Defoliation",
    11: "Southern Pine Beetle",
    12: "Insect, Disease, or Drought Stress",
    13: "Other Loss",
    14: "Vegetation Successional Growth",
    15: "Stable",
    16: "Non-Processing Area Mask",
}

print(f"Loaded {len(CHANGE_CLASSES)} LCMS change codes.")

Loaded 16 LCMS change codes.


#### Step 1B observations

All 16 LCMS annual change classes were mapped to readable labels for subsequent analysis.

The classes span stable conditions, vegetation growth, wildfire, anthropogenic vegetation removal, and several natural disturbance processes. No class is automatically treated as contamination at this stage.

`Tree Removal` and `Mechanical Land Transformation` are particularly relevant candidates for identifying anthropogenic disturbance, while the focal `Wildfire` class and `Vegetation Successional Growth` require different treatment because they may represent the event and recovery process of interest.

### Step 1C — Verify the LCMS spatial grid and Highline coverage

A representative LCMS annual image is inspected to confirm its native spatial resolution, coordinate reference system, and data coverage across the Highline Fire perimeter.

LCMS is expected to operate at approximately 30 m resolution, comparable in nominal resolution to Landsat. This does not imply that its grid is identical to the Landsat grid used elsewhere in the project; all sources will later be summarized independently onto the common 90 m project grid.

In [4]:
def get_lcms_year(year):
    return ee.Image(
        lcms_highline
        .filter(ee.Filter.eq("year", year))
        .first()
    )

lcms_2017 = get_lcms_year(2017).select("Change")

projection = lcms_2017.projection()

print("LCMS CRS:", projection.crs().getInfo())
print("LCMS nominal scale (m):", projection.nominalScale().getInfo())

highline_area_km2 = highline_geom.area().divide(1e6).getInfo()

valid_area_m2 = (
    ee.Image.pixelArea()
    .updateMask(lcms_2017.mask())
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline_geom,
        scale=30,
        maxPixels=1e8,
        tileScale=4,
    )
    .get("area")
    .getInfo()
)

valid_area_km2 = valid_area_m2 / 1e6
coverage_pct = 100 * valid_area_km2 / highline_area_km2

print("Highline vector area (km²):", round(highline_area_km2, 2))
print("LCMS valid area (km²):", round(valid_area_km2, 2))
print("Approximate LCMS coverage (%):", round(coverage_pct, 2))

LCMS CRS: None
LCMS nominal scale (m): 30
Highline vector area (km²): 378.31
LCMS valid area (km²): 379.18
Approximate LCMS coverage (%): 100.23


#### Step 1C observations

The LCMS `Change` product has a nominal spatial resolution of 30 m and effectively complete coverage of the Highline Fire perimeter.

- LCMS nominal scale: 30 m
- Highline vector area: 378.31 km²
- LCMS valid raster area: 379.18 km²
- Approximate coverage: 100.23%

The slight difference between vector and raster area is consistent with raster boundary approximation at the requested scale and does not indicate a coverage problem.

Although LCMS and Landsat both have approximately 30 m native resolution, their grids are not assumed to be spatially identical. LCMS will later be summarized independently onto the common 90 m project grid.

## Step 2 — Audit the Highline LCMS disturbance history

### Step 2A — Summarize annual LCMS change classes from 2007 through 2022

The full Highline LCMS change history is examined from ten years before the focal wildfire through the fifth post-fire growing season.

For the proof of concept, 2007–2016 provides a deliberately broad pre-fire disturbance-history window. The final production lookback will later be selected by comparing shorter 5-, 7-, and 10-year rules across the full wildfire population.

Annual change-class area fractions are summarized across the Highline perimeter to identify the dominant types and timing of landscape change before, during, and after the 2017 focal wildfire.

In [5]:
import pandas as pd

def summarize_lcms_change(year):
    change = get_lcms_year(year).select("Change")

    grouped = (
        ee.Image.pixelArea()
        .addBands(change)
        .reduceRegion(
            reducer=ee.Reducer.sum().group(
                groupField=1,
                groupName="change_class"
            ),
            geometry=highline_geom,
            scale=30,
            maxPixels=1e8,
            tileScale=4,
        )
        .get("groups")
        .getInfo()
    )

    total_area = sum(item["sum"] for item in grouped)

    rows = []
    for item in grouped:
        code = int(item["change_class"])
        area_pct = 100 * item["sum"] / total_area

        rows.append({
            "year": year,
            "code": code,
            "class": CHANGE_CLASSES.get(code, "Unknown"),
            "area_pct": area_pct,
        })

    return rows


records = []

for year in range(2007, 2023):
    records.extend(summarize_lcms_change(year))

change_df = pd.DataFrame(records)

for year in range(2007, 2023):
    print(f"\n{year}")

    year_df = change_df[
        (change_df["year"] == year) &
        (change_df["area_pct"] >= 0.05)
    ].sort_values("area_pct", ascending=False)

    for _, row in year_df.iterrows():
        print(f"{row['class']:<35} {row['area_pct']:6.2f}%")


2007
Stable                               56.07%
Insect, Disease, or Drought Stress   27.55%
Vegetation Successional Growth       16.35%

2008
Stable                               56.93%
Insect, Disease, or Drought Stress   26.43%
Vegetation Successional Growth       16.51%
Wildfire                              0.13%

2009
Stable                               58.05%
Insect, Disease, or Drought Stress   25.74%
Vegetation Successional Growth       16.21%

2010
Stable                               58.70%
Insect, Disease, or Drought Stress   26.31%
Vegetation Successional Growth       14.98%

2011
Stable                               59.95%
Insect, Disease, or Drought Stress   25.50%
Vegetation Successional Growth       14.53%

2012
Stable                               60.64%
Insect, Disease, or Drought Stress   25.21%
Vegetation Successional Growth       14.14%

2013
Stable                               57.58%
Insect, Disease, or Drought Stress   28.62%
Vegetation Successional Growth    

#### Step 2A observations

The annual LCMS change history from 2007–2022 reveals a clear distinction between persistent ecological change processes and discrete disturbance events.

Before the Highline Fire, most of the perimeter is consistently classified as Stable, Insect/Disease/Drought Stress, or Vegetation Successional Growth. The persistent presence of stress and growth classes demonstrates that LCMS change classifications cannot be treated as a simple binary disturbance screen.

The Highline wildfire signal is concentrated primarily in the 2018 LCMS product rather than 2017, with approximately 47.6% of the perimeter classified as Wildfire in 2018. Because LCMS derives annual products from seasonal image composites and time-series change detection, the mapped LCMS year should not be assumed to provide an exact event date, particularly for disturbances occurring during the annual compositing season.

Following the fire signal, Vegetation Successional Growth becomes the dominant change class in 2019 and 2020 and remains substantial through 2022, consistent with post-fire vegetation recovery.

No large Tree Removal or Mechanical Land Transformation signal is apparent in the area-wide summaries. These results suggest that LCMS should be used selectively to identify meaningful competing disturbances rather than treating all non-stable change as contamination.

### Step 2B — Isolate potential competing disturbances

The area-wide LCMS history contains several classes that reflect normal ecological dynamics or the focal wildfire itself and therefore should not automatically trigger exclusion.

This step isolates less common change classes that could represent competing disturbances, including prescribed fire, mechanical land transformation, tree removal, wind, inundation, defoliation, insect outbreaks, and other loss.

Persistent `Insect, Disease, or Drought Stress` is retained as ecological context rather than treated as an automatic contamination flag. `Wildfire` is examined separately with MTBS, while `Stable` and `Vegetation Successional Growth` are not contamination classes.

Potential competing disturbances are quantified even when they affect less than 0.05% of the Highline perimeter.

In [6]:
candidate_codes = {
    1: "Wind",
    2: "Hurricane",
    5: "Inundation",
    6: "Prescribed Fire",
    8: "Mechanical Land Transformation",
    9: "Tree Removal",
    10: "Defoliation",
    11: "Southern Pine Beetle",
    13: "Other Loss",
}

candidate_df = change_df[
    change_df["code"].isin(candidate_codes.keys())
].copy()

candidate_df["area_km2"] = (
    candidate_df["area_pct"] / 100 * highline_area_km2
)

for year in range(2007, 2023):
    year_df = candidate_df[
        (candidate_df["year"] == year) &
        (candidate_df["area_pct"] > 0)
    ].sort_values("area_pct", ascending=False)

    if not year_df.empty:
        print(f"\n{year}")

        for _, row in year_df.iterrows():
            print(
                f"{row['class']:<35}"
                f"{row['area_pct']:8.4f}%   "
                f"{row['area_km2']:8.4f} km²"
            )


2007
Other Loss                           0.0184%     0.0696 km²

2008
Other Loss                           0.0042%     0.0158 km²

2009
Other Loss                           0.0012%     0.0044 km²
Inundation                           0.0001%     0.0004 km²

2010
Other Loss                           0.0023%     0.0088 km²

2011
Other Loss                           0.0242%     0.0914 km²
Inundation                           0.0003%     0.0010 km²

2012
Other Loss                           0.0067%     0.0252 km²

2013
Other Loss                           0.0043%     0.0164 km²

2014
Other Loss                           0.1548%     0.5855 km²
Inundation                           0.0003%     0.0012 km²

2015
Other Loss                           0.0027%     0.0101 km²
Inundation                           0.0002%     0.0006 km²

2016
Other Loss                           0.0130%     0.0492 km²
Inundation                           0.0005%     0.0020 km²

2018
Other Loss                        

#### Step 2B observations

The targeted competing-disturbance screen indicates that the Highline Fire perimeter contains very little LCMS change from the candidate disturbance classes.

No Tree Removal, Mechanical Land Transformation, Prescribed Fire, Wind, Defoliation, or Southern Pine Beetle classifications were detected during the 2007–2022 audit period.

Nearly all detected candidate change was classified as `Other Loss`, with only negligible amounts of `Inundation`.

Most annual `Other Loss` fractions were extremely small. The most notable years were:

- 2014: approximately 0.155% of the Highline perimeter (~0.59 km²)
- 2019: approximately 0.119% (~0.45 km²)
- 2021: approximately 1.164% (~4.41 km²)

The 2021 signal is the clear outlier and occurs during the post-fire recovery period before the Year-5 target. However, because `Other Loss` is an ambiguous LCMS category, it should be spatially investigated rather than automatically treated as contamination.

These fire-wide percentages are diagnostic only. Production disturbance screening will ultimately operate at the common 90 m patch level so that localized competing disturbances can affect individual patches without requiring exclusion of an otherwise usable focal fire.

### Step 2C — Spatially inspect notable `Other Loss` detections

The three largest `Other Loss` signals identified in the annual audit are mapped within the Highline Fire perimeter.

The objective is to determine whether these detections form coherent localized disturbances, occur repeatedly in the same locations, or appear as scattered isolated pixels.

Particular attention is given to 2021 because it contains the largest `Other Loss` fraction and occurs between the first post-fire season and the Year-5 target.

In [7]:
import geemap

m = geemap.Map()

m.centerObject(highline_geom, 10)

highline_fc = ee.FeatureCollection([highline])

perimeter = (
    ee.Image()
    .byte()
    .paint(
        featureCollection=highline_fc,
        color=1,
        width=2
    )
)

other_loss_2014 = (
    get_lcms_year(2014)
    .select("Change")
    .eq(13)
    .selfMask()
    .clip(highline_geom)
)

other_loss_2019 = (
    get_lcms_year(2019)
    .select("Change")
    .eq(13)
    .selfMask()
    .clip(highline_geom)
)

other_loss_2021 = (
    get_lcms_year(2021)
    .select("Change")
    .eq(13)
    .selfMask()
    .clip(highline_geom)
)

m.addLayer(other_loss_2014, {"palette": ["yellow"]}, "Other Loss — 2014")
m.addLayer(other_loss_2019, {"palette": ["orange"]}, "Other Loss — 2019")
m.addLayer(other_loss_2021, {"palette": ["red"]}, "Other Loss — 2021")

m.addLayer(perimeter, {"palette": ["black"]}, "Highline perimeter")

m

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

#### Step 2C observations

Spatial inspection shows that the notable `Other Loss` detections are localized rather than distributed uniformly across the Highline perimeter.

- The 2014 detection is concentrated in a small northern cluster.
- The 2019 detection is very small and localized in the southern portion of the perimeter.
- The larger 2021 signal forms several spatially coherent clusters across the central and southern portions of Highline.

The 2021 pattern therefore appears to represent a meaningful detected fast-loss signal rather than only isolated classification noise. However, LCMS `Other Loss` does not identify a specific disturbance cause. It represents Fast Loss for which none of the more specific LCMS disturbance classes could be assigned.

Consequently, `Other Loss` should not be interpreted automatically as anthropogenic contamination. In a post-fire landscape, some such detections could represent ecological processes that are legitimately part of the recovery trajectory. `Other Loss` will therefore be retained as an ambiguity/QA signal rather than used as an automatic exclusion criterion at this stage.

## Step 3 — Define provisional disturbance-screening logic

### Step 3A — Group LCMS classes by their role in the recovery study

LCMS change classes are grouped according to how they should provisionally be interpreted for wildfire-recovery quality control.

The grouping distinguishes clear competing disturbances from ambiguous ecological change. This avoids treating every detected vegetation change as contamination and reduces the risk of excluding legitimate poor-recovery outcomes.

These categories remain provisional until they can be audited across the full wildfire population in Phase 3.

In [8]:
LCMS_GROUPS = {
    # Strong candidates for excluding an affected patch
    "clear_competing_disturbance": {
        1,   # Wind
        2,   # Hurricane
        5,   # Inundation
        6,   # Prescribed Fire
        8,   # Mechanical Land Transformation
        9,   # Tree Removal
        10,  # Defoliation
        11,  # Southern Pine Beetle
    },

    # Important signal, but cause/interpretation is ambiguous
    "ambiguous_loss": {
        13,  # Other Loss
    },

    # Ecological/environmental processes not automatically excluded
    "ecological_context": {
        3,   # Snow or Ice Transition
        4,   # Desiccation
        12,  # Insect, Disease, or Drought Stress
    },

    # Expected/non-contaminating trajectory states
    "expected": {
        14,  # Vegetation Successional Growth
        15,  # Stable
    },

    # Wildfire handled separately using focal-fire identity + MTBS
    "wildfire": {
        7,
    },

    # Data availability / QA
    "nonprocessing": {
        16,
    },
}

for group, codes in LCMS_GROUPS.items():
    labels = [CHANGE_CLASSES[c] for c in sorted(codes)]
    print(f"{group}:")
    for label in labels:
        print(f"  - {label}")

clear_competing_disturbance:
  - Wind
  - Hurricane
  - Inundation
  - Prescribed Fire
  - Mechanical Land Transformation
  - Tree Removal
  - Defoliation
  - Southern Pine Beetle
ambiguous_loss:
  - Other Loss
ecological_context:
  - Snow or Ice Transition
  - Desiccation
  - Insect, Disease, or Drought Stress
expected:
  - Vegetation Successional Growth
  - Stable
wildfire:
  - Wildfire
nonprocessing:
  - Non-Processing Area Mask


#### Step 3A observations

The 16 LCMS change classes were grouped into six project-specific interpretation categories.

Clear competing-disturbance candidates include discrete events such as Tree Removal, Mechanical Land Transformation, Prescribed Fire, Wind, Inundation, Defoliation, and Southern Pine Beetle.

`Other Loss` is retained separately because it indicates detected fast vegetation loss without a confidently assigned cause.

Desiccation and Insect/Disease/Drought Stress are treated as ecological context rather than automatic contamination, while Stable and Vegetation Successional Growth represent expected landscape states or recovery.

Wildfire is handled separately because the focal wildfire is the event of interest, while additional wildfire disturbance is more reliably identified using MTBS.

These groupings are provisional and will be evaluated across the full wildfire population before final exclusion rules are locked.

### Step 3B — Define temporal disturbance-screening windows

LCMS disturbance screening is separated into pre-fire and post-fire periods relative to each focal wildfire.

For the Highline Fire:

- 2007–2016 forms a broad 10-year pre-fire disturbance-history audit.
- 2017 is the focal wildfire year.
- 2018–2022 covers the five complete post-fire growing seasons through the Year-5 target.

The pre-fire audit window is intentionally broader than the three years used to calculate `NBRpre`. This allows recent disturbances occurring before the NBR baseline years to be detected if they could still influence the pre-fire ecological state.

The final production lookback will later compare 5-, 7-, and 10-year alternatives rather than fixing the rule using Highline alone.

Post-fire disturbance screening continues through Year 5 because any competing disturbance during the recovery interval could affect the interpretation of the target.

In [9]:
FIRE_YEAR = 2017
YEAR5 = 2022

PRE_FIRE_LOOKBACK_POC = 10

prefire_years = list(
    range(FIRE_YEAR - PRE_FIRE_LOOKBACK_POC, FIRE_YEAR)
)

focal_fire_year = FIRE_YEAR

postfire_years = list(
    range(FIRE_YEAR + 1, YEAR5 + 1)
)

print("Pre-fire disturbance audit:", prefire_years)
print("Focal fire year:", focal_fire_year)
print("Post-fire through Year 5:", postfire_years)

print("\nNBRpre years:", [2014, 2015, 2016])
print("NBR1 year:", 2018)
print("NBR5 year:", 2022)

Pre-fire disturbance audit: [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
Focal fire year: 2017
Post-fire through Year 5: [2018, 2019, 2020, 2021, 2022]

NBRpre years: [2014, 2015, 2016]
NBR1 year: 2018
NBR5 year: 2022


#### Step 3B observations

For Highline, LCMS is audited across:

- 2007–2016: broad 10-year pre-fire disturbance history
- 2017: focal wildfire year
- 2018–2022: post-fire period through Year 5

The three `NBRpre` years are 2014–2016, while the broader pre-fire LCMS window checks whether earlier recent disturbances may have affected that baseline.

### Step 3C — Define patch-level LCMS QA fields

LCMS will contribute a small set of disturbance-quality-control fields for each future 90 m patch.

The fields distinguish clear competing disturbances from ambiguous loss and separate pre-fire from post-fire timing. Final exclusion thresholds will be chosen later using the full fire population.

In [10]:
LCMS_QA_FIELDS = {
    "prefire_clear_disturbance_fraction": (
        "Fraction of the patch affected by a clear competing disturbance "
        "during the selected pre-fire lookback."
    ),

    "prefire_ambiguous_loss_fraction": (
        "Fraction of the patch classified as Other Loss during the "
        "selected pre-fire lookback."
    ),

    "postfire_clear_disturbance_fraction": (
        "Fraction of the patch affected by a clear competing disturbance "
        "after the focal fire through Year 5."
    ),

    "postfire_ambiguous_loss_fraction": (
        "Fraction of the patch classified as Other Loss after the focal "
        "fire through Year 5."
    ),

    "postfire_clear_disturbance_year": (
        "First post-fire year with a clear competing disturbance, if any."
    ),

    "postfire_ambiguous_loss_year": (
        "First post-fire year with Other Loss, if any."
    ),
}

for field, description in LCMS_QA_FIELDS.items():
    print(f"{field}")
    print(f"  {description}\n")

prefire_clear_disturbance_fraction
  Fraction of the patch affected by a clear competing disturbance during the selected pre-fire lookback.

prefire_ambiguous_loss_fraction
  Fraction of the patch classified as Other Loss during the selected pre-fire lookback.

postfire_clear_disturbance_fraction
  Fraction of the patch affected by a clear competing disturbance after the focal fire through Year 5.

postfire_ambiguous_loss_fraction
  Fraction of the patch classified as Other Loss after the focal fire through Year 5.

postfire_clear_disturbance_year
  First post-fire year with a clear competing disturbance, if any.

postfire_ambiguous_loss_year
  First post-fire year with Other Loss, if any.



#### Step 3C observations

LCMS will contribute QA/filtering metadata rather than model predictors.

For each 90 m patch, LCMS will summarize the fraction affected by clear competing disturbances or ambiguous `Other Loss`, along with relevant timing. These fields can then be used to determine whether the entire patch row should be retained or excluded.

Individual 30 m LCMS pixels will not be removed from an otherwise fixed 90 m patch. Disturbance fractions will instead describe how much of that patch was affected. Final thresholds will be selected during the full-population audit.

## Step 4 — Validate the provisional LCMS screen

### Step 4A — Apply disturbance groups across the Highline timeline

Window-level masks are created for clear competing disturbances and ambiguous `Other Loss`.

A location is counted once if it was affected in any year within the window:

- 2007–2016: broad pre-fire audit
- 2017–2022: focal-fire year through Year 5

The focal `Wildfire` class is excluded from the competing-disturbance mask.

In [11]:
CLEAR_CODES = sorted(LCMS_GROUPS["clear_competing_disturbance"])
AMBIGUOUS_CODES = sorted(LCMS_GROUPS["ambiguous_loss"])

prefire_audit_years = list(range(2007, 2017))
recovery_qa_years = list(range(2017, 2023))


def any_change_in_window(years, codes):
    masks = []

    for year in years:
        change = get_lcms_year(year).select("Change")

        mask = change.remap(
            codes,
            [1] * len(codes),
            0
        ).rename("flag")

        masks.append(mask)

    return ee.ImageCollection.fromImages(masks).max()


prefire_clear = any_change_in_window(
    prefire_audit_years, CLEAR_CODES
)

prefire_ambiguous = any_change_in_window(
    prefire_audit_years, AMBIGUOUS_CODES
)

recovery_clear = any_change_in_window(
    recovery_qa_years, CLEAR_CODES
)

recovery_ambiguous = any_change_in_window(
    recovery_qa_years, AMBIGUOUS_CODES
)


lcms_reference = get_lcms_year(2017).select("Change")

valid_area = (
    ee.Image.pixelArea()
    .updateMask(lcms_reference.mask())
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline_geom,
        scale=30,
        maxPixels=1e8,
        tileScale=4,
    )
    .get("area")
)


def affected_fraction(mask):
    affected_area = (
        ee.Image.pixelArea()
        .updateMask(mask.eq(1))
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=highline_geom,
            scale=30,
            maxPixels=1e8,
            tileScale=4,
        )
        .get("area")
    )

    return ee.Number(affected_area).divide(valid_area).multiply(100)


print(
    "Pre-fire clear disturbance:",
    round(affected_fraction(prefire_clear).getInfo(), 4),
    "%"
)

print(
    "Pre-fire ambiguous loss:",
    round(affected_fraction(prefire_ambiguous).getInfo(), 4),
    "%"
)

print(
    "2017–2022 clear disturbance:",
    round(affected_fraction(recovery_clear).getInfo(), 4),
    "%"
)

print(
    "2017–2022 ambiguous loss:",
    round(affected_fraction(recovery_ambiguous).getInfo(), 4),
    "%"
)

Pre-fire clear disturbance: 0.0011 %
Pre-fire ambiguous loss: 0.2298 %
2017–2022 clear disturbance: 0.001 %
2017–2022 ambiguous loss: 1.2892 %


#### Step 4A observations

Clear competing disturbances are effectively absent across Highline:

- Pre-fire clear disturbance: 0.0011%
- 2017–2022 clear disturbance: 0.0010%

Ambiguous `Other Loss` affects approximately 0.23% of the broad pre-fire audit area and 1.29% of the recovery-period area.

The post-fire value exceeds any individual annual fraction because it represents the union of locations flagged at least once across 2017–2022.

### Step 4B — Compare candidate pre-fire lookback windows

Five-, seven-, and ten-year pre-fire disturbance windows are compared for Highline.

This is a sensitivity check only. The final lookback length will be selected from the full fire population in Phase 3.

In [12]:
lookbacks = [5, 7, 10]

for lookback in lookbacks:
    years = list(range(FIRE_YEAR - lookback, FIRE_YEAR))

    clear_mask = any_change_in_window(
        years, CLEAR_CODES
    )

    ambiguous_mask = any_change_in_window(
        years, AMBIGUOUS_CODES
    )

    clear_pct = affected_fraction(clear_mask).getInfo()
    ambiguous_pct = affected_fraction(ambiguous_mask).getInfo()

    print(f"\n{lookback}-year lookback ({years[0]}–{years[-1]})")
    print(f"Clear disturbance: {clear_pct:.4f}%")
    print(f"Ambiguous loss:    {ambiguous_pct:.4f}%")


5-year lookback (2012–2016)
Clear disturbance: 0.0010%
Ambiguous loss:    0.1799%

7-year lookback (2010–2016)
Clear disturbance: 0.0010%
Ambiguous loss:    0.2061%

10-year lookback (2007–2016)
Clear disturbance: 0.0011%
Ambiguous loss:    0.2298%


#### Step 4B observations

Highline is minimally sensitive to the candidate pre-fire lookback length.

Clear competing disturbance remains approximately 0.001% under 5-, 7-, and 10-year windows. Ambiguous loss increases only modestly from 0.180% to 0.230% as the lookback expands.

The final pre-fire lookback will therefore be selected using population-level evidence in Phase 3 rather than the Highline POC.

### Step 4C — Audit annual-timing edge cases

LCMS annual classifications are not treated as exact disturbance dates.

The Highline wildfire occurred in 2017, yet its dominant LCMS wildfire signal appears in 2018. Similarly, a disturbance labeled in the Year-5 calendar year may occur before or after the Landsat target window.

These timing ambiguities must be retained in the production QA logic.

In [13]:
def class_fraction(year, code):
    year_df = change_df[
        (change_df["year"] == year) &
        (change_df["code"] == code)
    ]

    if year_df.empty:
        return 0.0

    return float(year_df["area_pct"].iloc[0])


print("Wildfire fraction:")
print(f"2017: {class_fraction(2017, 7):.4f}%")
print(f"2018: {class_fraction(2018, 7):.4f}%")

print("\nYear-5 (2022) QA signals:")
print(
    f"Clear competing disturbance: "
    f"{affected_fraction(any_change_in_window([2022], CLEAR_CODES)).getInfo():.4f}%"
)
print(
    f"Ambiguous Other Loss: "
    f"{affected_fraction(any_change_in_window([2022], AMBIGUOUS_CODES)).getInfo():.4f}%"
)

Wildfire fraction:
2017: 0.5657%
2018: 47.6386%

Year-5 (2022) QA signals:
Clear competing disturbance: 0.0005%
Ambiguous Other Loss: 0.0025%


#### Step 4C observations

Highline's focal wildfire occurred in 2017, but the dominant LCMS wildfire classification appears in 2018 (47.64% versus 0.57% in 2017). LCMS annual labels therefore should not be interpreted as exact disturbance dates.

Potential Year-5 timing ambiguity is negligible for Highline: only 0.0005% clear competing disturbance and 0.0025% `Other Loss` are detected in 2022.

Production QA will retain these timing limitations explicitly.

## Step 5 — Lock and package the LCMS proof of concept

### Step 5A — Provisional production contract

LCMS will serve as patch-level disturbance QA rather than as a predictive feature source.

Production logic will:

- summarize LCMS disturbance classes onto each common 90 m patch,
- distinguish clear competing disturbances from ambiguous `Other Loss`,
- audit a recent pre-fire disturbance window and the full focal-fire-to-Year-5 period,
- retain disturbance fractions and timing as QA metadata,
- handle additional wildfire disturbance primarily with MTBS,
- and avoid using LCMS QA fields as model predictors.

Final 5-, 7-, or 10-year pre-fire lookback and patch-exclusion thresholds will be selected from full-population evidence in Phase 3.

In [14]:
LCMS_PRODUCTION_CONTRACT = {
    "role": "QA/filtering metadata; not model predictors",

    "spatial_unit": "summarize native LCMS pixels onto common 90 m patches",

    "prefire_lookback": "compare 5, 7, and 10 years in Phase 3",

    "clear_competing_disturbance": (
        "retain fraction/timing; evaluate patch exclusion threshold in Phase 3"
    ),

    "ambiguous_other_loss": (
        "retain as QA flag/fraction; do not automatically exclude"
    ),

    "wildfire": (
        "focal wildfire is allowed; prior fire/reburn handled primarily with MTBS"
    ),

    "postfire_window": (
        "audit continuously from focal fire year through Year 5"
    ),

    "year5_timing": (
        "annual LCMS classification may be ambiguous relative to Jun-Sep NBR5"
    ),
}

for key, value in LCMS_PRODUCTION_CONTRACT.items():
    print(f"{key}: {value}")

role: QA/filtering metadata; not model predictors
spatial_unit: summarize native LCMS pixels onto common 90 m patches
prefire_lookback: compare 5, 7, and 10 years in Phase 3
clear_competing_disturbance: retain fraction/timing; evaluate patch exclusion threshold in Phase 3
ambiguous_other_loss: retain as QA flag/fraction; do not automatically exclude
wildfire: focal wildfire is allowed; prior fire/reburn handled primarily with MTBS
postfire_window: audit continuously from focal fire year through Year 5
year5_timing: annual LCMS classification may be ambiguous relative to Jun-Sep NBR5


#### Patch-level disturbance logic

For each 90 m patch, LCMS disturbance is summarized as a spatial union across the relevant years.

A location is counted once if it was affected at least once during the window, regardless of how many years the disturbance persisted. Repeated detections at the same location are therefore not double-counted.

The resulting fraction represents the proportion of the 90 m patch affected by a competing disturbance at any point during the window. Final patch-exclusion thresholds will be selected during the Phase 3 population-level audit.

### Step 5B — Save the LCMS validation artifact

The 2021 `Other Loss` map is retained as the primary LCMS validation artifact because it shows the largest ambiguous-loss signal detected during the Highline recovery period.

In [15]:
from pathlib import Path

repo_root = (
    Path.cwd()
    if (Path.cwd() / "pyproject.toml").exists()
    else Path.cwd().parent
)

lcms_fig_dir = repo_root / "outputs" / "figures" / "lcms"
lcms_fig_dir.mkdir(parents=True, exist_ok=True)

print(lcms_fig_dir)

outputs\figures\lcms


In [16]:
import urllib.request

output_path = lcms_fig_dir / "highline_2021_other_loss.png"

# White background inside the Highline perimeter
background = (
    ee.Image.constant(1)
    .clip(highline_geom)
    .visualize(min=0, max=1, palette=["ffffff"])
)

# 2021 Other Loss in red
other_loss_vis = (
    other_loss_2021
    .visualize(min=0, max=1, palette=["ff0000"])
)

# Highline perimeter in black
perimeter_vis = perimeter.visualize(
    min=0,
    max=1,
    palette=["000000"]
)

figure = (
    background
    .blend(other_loss_vis)
    .blend(perimeter_vis)
)

url = figure.getThumbURL({
    "region": highline_geom.bounds(),
    "dimensions": 1200,
    "format": "png",
})

urllib.request.urlretrieve(url, output_path)

print("Saved:", output_path)
print("Exists:", output_path.exists())
print("Size (KB):", round(output_path.stat().st_size / 1024, 1))

Saved: outputs\figures\lcms\highline_2021_other_loss.png
Exists: True
Size (KB): 42.0


#### Step 5B observations

The 2021 `Other Loss` validation map was saved successfully.

The detections form multiple localized clusters within the Highline perimeter, confirming that the 2021 signal is spatially coherent rather than uniformly distributed noise. Because `Other Loss` does not identify a specific cause, it remains an ambiguity/QA signal rather than an automatic exclusion criterion.

In [17]:
expected_figure = lcms_fig_dir / "highline_2021_other_loss.png"

print("LCMS change classes:", len(CHANGE_CLASSES))
print("LCMS interpretation groups:", len(LCMS_GROUPS))
print("LCMS QA fields:", len(LCMS_QA_FIELDS))

print(
    "Highline audit years:",
    f"{change_df['year'].min()}–{change_df['year'].max()}"
)

print("Validation figure:", "FOUND" if expected_figure.exists() else "MISSING")

LCMS change classes: 16
LCMS interpretation groups: 6
LCMS QA fields: 6
Highline audit years: 2007–2022
Validation figure: FOUND
